<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/07-joins-deep-dive/02-broadcast-hint-and-threshold.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 7.2 — broadcast() and autoBroadcastJoinThreshold

We watch the planner's size ESTIMATE — the thing the threshold
actually compares — diverge from disk size and from reality, then
use hints to take the decision back. Writes parquet to a local
`output/` dir; the last cell cleans it up.

In [ ]:
import os, sys, shutil
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, broadcast

spark = (
    SparkSession.builder
    .appName("7.2-broadcast-threshold")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("threshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))  # 10 MB

def plan_size_estimate(df):
    """The planner's sizeInBytes stat for a DataFrame — the number the
    broadcast threshold is compared against. Internal API: diagnostics only."""
    return int(df._jdf.queryExecution().optimizedPlan().stats().sizeInBytes())

## Setup: a fact and a dimension on disk

Estimates start from FILE SIZES, so the tables must actually live on
disk to make this honest (a createDataFrame in memory has different stats).

In [ ]:
out = "output"
facts_mem = spark.range(0, 1_000_000).select(
    col("id").alias("order_id"),
    (col("id") % 5000).alias("cust_key"),
    (F.rand(seed=42) * 100).alias("amount"),
)
dim_mem = spark.range(0, 5000).select(
    col("id").alias("cust_key"),
    F.concat(F.lit("cust_"), col("id").cast("string")).alias("name"),
    (col("id") % 100 == 0).alias("active"),   # 1% active — the filter demo
)
facts_mem.write.mode("overwrite").parquet(f"{out}/facts")
dim_mem.write.mode("overwrite").parquet(f"{out}/dim")

facts = spark.read.parquet(f"{out}/facts")
dim = spark.read.parquet(f"{out}/dim")

def dir_bytes(path):
    return sum(os.path.getsize(os.path.join(r, f))
               for r, _, fs in os.walk(path) for f in fs)

print(f"dim on disk:   {dir_bytes(f'{out}/dim'):>10,} bytes (snappy parquet)")
print(f"dim estimated: {plan_size_estimate(dim):>10,} bytes (planner sizeInBytes)")

## Under the threshold: auto-broadcast fires

In [ ]:
big_join = facts.join(dim, "cust_key")
big_join.explain()   # BroadcastHashJoin, BuildRight — nobody hinted anything

## Lie #1 — filters skew estimates HIGH

`active` keeps 1% of rows (50 of 5000). Without column stats the
planner can't know that; watch how little the estimate moves — and
with a small enough threshold, the filtered dim misses the cut.

In [ ]:
active_dim = dim.filter(col("active"))
print("rows:      ", dim.count(), "->", active_dim.count())
print("estimate:  ", plan_size_estimate(dim), "->", plan_size_estimate(active_dim))

In [ ]:
# Shrink the threshold below the (inflated) estimate to simulate the
# production version of this: a genuinely-small side that misses the cut.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(8 * 1024))
facts.join(active_dim, "cust_key").explain()             # SortMergeJoin — the silent slow path

In [ ]:
# You know what the planner can't. Take the decision back:
facts.join(broadcast(active_dim), "cust_key").explain()  # BroadcastHashJoin again

## Lie #2 — compression skews estimates LOW

The threshold compares against ~disk-derived sizes; snappy parquet
with repetitive columns can be several times smaller on disk than in
memory. Nothing to "run" here — the point is the ratio you just
printed in the setup cell. On real wide dimensions the in-memory
inflation is routinely 5–10x, which is why "under 10 MB" is not the
same as "safe to hold N copies of".

## A hint is a request, not a command

Hint the PRESERVED side of an outer join (illegal per 7.1) and Spark
ignores it silently. explain() is the only way you'd ever know.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # manual mode from here on

facts.join(broadcast(dim), "cust_key", "left").explain()   # legal: BHJ BuildRight
facts.join(broadcast(dim), "cust_key", "full").explain()   # ignored: SortMergeJoin, no warning

## The production stance: threshold off, broadcasts explicit

In [ ]:
# threshold is -1 now: nothing broadcasts unless the code says so
enriched = (
    facts
    .join(broadcast(dim), "cust_key", "left")
    .withColumn("revenue_band", (col("amount") / 25).cast("int"))
)
enriched.explain()
enriched.groupBy("revenue_band").count().orderBy("revenue_band").show()

## Exercises

1. Find the break-even: bisect `spark.range` sizes for the dimension
   until auto-broadcast (threshold back at 10 MB) stops firing. How
   many ROWS is 10 MB of this schema, per the planner?
2. `dim.cache()` then re-check `plan_size_estimate(dim)` after an
   action. Cached plans carry different (in-memory) stats — does the
   broadcast decision change?
3. Both sides hinted: `broadcast(a).join(broadcast(b), ...)` — which
   one wins? Check with two tables of clearly different sizes.
4. Simulate the timeout failure WITHOUT actually failing: read the
   docs for `spark.sql.broadcastTimeout`, then write (don't run) the
   conf + code for the incident writeup: "dim grew from 8 MB to 3 GB;
   job now dies at 300s". What are the two fixes, and which is right?

In [ ]:
# cleanup — this notebook wrote local parquet
spark.stop()
shutil.rmtree(out, ignore_errors=True)
print("cleaned", out)